In [13]:
import sys
sys.path.append(r'N:\DE\CC Transaction Pipeline')
import cfg.config as config

from pyspark.sql import SparkSession
import os

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType, FloatType, DoubleType
import pyspark.sql.functions as F

os.environ['HADOOP_HOME'] = r'C:\Users\nisha\hadoop'

spark = SparkSession.builder \
    .appName("CCPipeline") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.4.2,com.amazonaws:aws-java-sdk-bundle:1.12.367") \
    .config("spark.hadoop.fs.s3a.access.key", config.ACCESS_KEY_ID) \
    .config("spark.hadoop.fs.s3a.secret.key", config.ACCESS_KEY_SECRET) \
    .config("spark.hadoop.fs.s3a.fast.upload.buffer", "bytebuffer") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()


Dim Customer

In [17]:
slv_dim_customer = spark.read \
        .format('parquet') \
        .load('s3a://cc-transaction-pipeline/brz/brz_dim_customer/')

slv_dim_customer.show()

+-----------+---------+------+--------------------+--------------------+-----+-----+-------+------------------+--------+--------------------+----------+--------------------+
|      first|     last|gender|              street|                city|state|  zip|    lat|              long|city_pop|                 job|       dob|           cust_uuid|
+-----------+---------+------+--------------------+--------------------+-----+-----+-------+------------------+--------+--------------------+----------+--------------------+
|   Jennifer|    Banks|     F|      561 Perry Cove|      Moravian Falls|   NC|28654|36.0788|          -81.1781|    3495|Psychologist, cou...|1988-03-09|8df71411-7ed9-484...|
|  Stephanie|     Gill|     F|43039 Riley Green...|              Orient|   WA|99160|48.8878|         -118.2105|     149|Special education...|1978-06-21|5d0d6615-62a5-443...|
|     Edward|  Sanchez|     M|594 White Dale Su...|          Malad City|   ID|83252|42.1808|          -112.262|    4154|Nature con

In [18]:
slv_dim_customer.printSchema()

root
 |-- first: string (nullable = true)
 |-- last: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- street: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- lat: string (nullable = true)
 |-- long: string (nullable = true)
 |-- city_pop: string (nullable = true)
 |-- job: string (nullable = true)
 |-- dob: string (nullable = true)
 |-- cust_uuid: string (nullable = true)



In [19]:
slv_dim_customer = slv_dim_customer.withColumn('zip',F.col('zip').cast(IntegerType()))
slv_dim_customer = slv_dim_customer.withColumn('lat',F.col('lat').cast(DoubleType()))
slv_dim_customer = slv_dim_customer.withColumn('long',F.col('long').cast(DoubleType()))
slv_dim_customer = slv_dim_customer.withColumn('city_pop',F.col('city_pop').cast(IntegerType()))
slv_dim_customer = slv_dim_customer.withColumn('dob',F.col('dob').cast(DateType()))

In [20]:
slv_dim_customer.printSchema()

root
 |-- first: string (nullable = true)
 |-- last: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- street: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip: integer (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- city_pop: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- dob: date (nullable = true)
 |-- cust_uuid: string (nullable = true)



In [ ]:
#Checking Nulls
slv_dim_customer.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in slv_dim_customer.columns]).show()

+-----+----+------+------+----+-----+---+---+----+--------+---+---+---------+
|first|last|gender|street|city|state|zip|lat|long|city_pop|job|dob|cust_uuid|
+-----+----+------+------+----+-----+---+---+----+--------+---+---+---------+
|    0|   0|     0|     0|   0|    0|  0|  0|   0|       0|  0|  0|        0|
+-----+----+------+------+----+-----+---+---+----+--------+---+---+---------+



In [23]:
slv_dim_customer.select('gender').distinct().show()

+------+
|gender|
+------+
|     F|
|     M|
+------+



In [24]:
slv_dim_customer.select('state').distinct().show()

+-----+
|state|
+-----+
|   AZ|
|   SC|
|   LA|
|   MN|
|   NJ|
|   DC|
|   OR|
|   VA|
|   RI|
|   WY|
|   KY|
|   NH|
|   MI|
|   NV|
|   WI|
|   ID|
|   CA|
|   NE|
|   CT|
|   MT|
+-----+
only showing top 20 rows


In [25]:
slv_dim_customer.dropDuplicates().count()

999

In [26]:
slv_dim_customer.write \
    .format('parquet') \
    .mode('overwrite') \
    .save('s3a://cc-transaction-pipeline/slv/slv_dim_customer/')

Dim Merchant

In [27]:
slv_dim_merchant = spark.read \
        .format('parquet') \
        .load('s3a://cc-transaction-pipeline/brz/brz_dim_merchant/')

slv_dim_merchant.show()

+--------------------+------------------+------------------+--------------------+
|            merchant|         merch_lat|        merch_long|          merch_uuid|
+--------------------+------------------+------------------+--------------------+
| fraud_Abbott-Rogahn|         26.460232|        -83.062399|f55a4f80-f9d2-41c...|
|fraud_Abbott-Steuber|         31.218339|        -81.606691|4dfca935-cd0b-42a...|
|fraud_Abernathy a...|          45.38473|       -121.637729|8aa32121-aa16-471...|
|   fraud_Abshire PLC|          37.62768|         -96.24989|ea16e3e8-8ef9-447...|
|fraud_Adams, Kova...|         49.807705|       -117.388555|6da32252-8d24-49c...|
| fraud_Adams-Barrows|         47.334898|        -85.664218|870dc211-14c1-48b...|
|fraud_Altenwerth,...|         34.625988|        -91.983465|15366c36-5c49-430...|
|fraud_Altenwerth-...|         45.716706|       -119.886246|352ffc57-3c10-487...|
| fraud_Ankunding LLC|29.889152000000003|        -97.003361|23929047-7dbe-474...|
|fraud_Ankunding

In [29]:
slv_dim_merchant.printSchema()

root
 |-- merchant: string (nullable = true)
 |-- merch_lat: string (nullable = true)
 |-- merch_long: string (nullable = true)
 |-- merch_uuid: string (nullable = true)



In [30]:
slv_dim_merchant = slv_dim_merchant.withColumn('merch_lat',F.col('merch_lat').cast(DoubleType()))
slv_dim_merchant = slv_dim_merchant.withColumn('merch_long',F.col('merch_long').cast(DoubleType()))

In [31]:
slv_dim_merchant.printSchema()

root
 |-- merchant: string (nullable = true)
 |-- merch_lat: double (nullable = true)
 |-- merch_long: double (nullable = true)
 |-- merch_uuid: string (nullable = true)



In [35]:
slv_dim_merchant.count()

693

In [36]:
slv_dim_merchant.dropDuplicates().count()

693

In [37]:
#Checking Nulls
slv_dim_merchant.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in slv_dim_merchant.columns]).show()

+--------+---------+----------+----------+
|merchant|merch_lat|merch_long|merch_uuid|
+--------+---------+----------+----------+
|       0|        0|         0|         0|
+--------+---------+----------+----------+



In [39]:
slv_dim_merchant.select('merchant').distinct().show(truncate=False)

+-------------------------------------+
|merchant                             |
+-------------------------------------+
|fraud_Bradtke, Torp and Bahringer    |
|fraud_Herman Inc                     |
|fraud_O'Hara-Wilderman               |
|fraud_Rau and Sons                   |
|fraud_Thiel PLC                      |
|fraud_Altenwerth, Cartwright and Koss|
|fraud_Effertz LLC                    |
|fraud_Greenholt Ltd                  |
|fraud_Ledner, Hartmann and Feest     |
|fraud_Robel, Cummerata and Prosacco  |
|fraud_Waelchi-Wolf                   |
|fraud_Douglas, DuBuque and McKenzie  |
|fraud_Hills-Boyer                    |
|fraud_Jast Ltd                       |
|fraud_Kihn-Schuster                  |
|fraud_Smitham-Schiller               |
|fraud_Gottlieb-Hansen                |
|fraud_Kerluke-Abshire                |
|fraud_Rippin-VonRueden               |
|fraud_Stroman, Hudson and Erdman     |
+-------------------------------------+
only showing top 20 rows


In [40]:
slv_dim_merchant = slv_dim_merchant.withColumn('merchant',F.regexp_replace("merchant", "fraud_", " "))

In [41]:
slv_dim_merchant.show()

+--------------------+------------------+------------------+--------------------+
|            merchant|         merch_lat|        merch_long|          merch_uuid|
+--------------------+------------------+------------------+--------------------+
|       Abbott-Rogahn|         26.460232|        -83.062399|f55a4f80-f9d2-41c...|
|      Abbott-Steuber|         31.218339|        -81.606691|4dfca935-cd0b-42a...|
|  Abernathy and Sons|          45.38473|       -121.637729|8aa32121-aa16-471...|
|         Abshire PLC|          37.62768|         -96.24989|ea16e3e8-8ef9-447...|
| Adams, Kovacek a...|         49.807705|       -117.388555|6da32252-8d24-49c...|
|       Adams-Barrows|         47.334898|        -85.664218|870dc211-14c1-48b...|
| Altenwerth, Cart...|         34.625988|        -91.983465|15366c36-5c49-430...|
|  Altenwerth-Kilback|         45.716706|       -119.886246|352ffc57-3c10-487...|
|       Ankunding LLC|29.889152000000003|        -97.003361|23929047-7dbe-474...|
|   Ankunding-Ca

In [42]:
slv_dim_merchant.write \
    .format('parquet') \
    .mode('overwrite') \
    .save('s3a://cc-transaction-pipeline/slv/slv_dim_merchant/')

Dim Card

In [53]:
slv_dim_card = spark.read \
        .format('parquet') \
        .load('s3a://cc-transaction-pipeline/brz/brz_dim_card/')

slv_dim_card.show()

+-------------------+--------------------+-------------+---------------+----------+------------+
|             cc_num|       card_provider|card_category|expiration_date|issue_date|credit_limit|
+-------------------+--------------------+-------------+---------------+----------+------------+
|   2703186189652095|    American Express|    Signature|     2030-02-24|2025-02-25|       45000|
|       630423337322|       VISA 19 digit|         Gold|     2028-04-02|2024-04-03|       15000|
|     38859492057661|        JCB 16 digit|         Gold|     2026-08-26|2021-08-27|        6000|
|   3534093764340240|             Maestro|     Platinum|     2025-09-10|2022-09-11|       27500|
|    375534208663984|        JCB 15 digit|     Infinite|     2028-08-03|2024-08-04|      100000|
|   4767265376804500|       VISA 16 digit|     Infinite|     2026-06-03|2022-06-04|       90000|
|     30074693890476|       VISA 13 digit|         Gold|     2027-12-04|2022-12-05|        7000|
|   6011360759745864|Diners Cl

In [54]:
slv_dim_card.printSchema()

root
 |-- cc_num: string (nullable = true)
 |-- card_provider: string (nullable = true)
 |-- card_category: string (nullable = true)
 |-- expiration_date: string (nullable = true)
 |-- issue_date: string (nullable = true)
 |-- credit_limit: string (nullable = true)



In [55]:
# slv_dim_card = slv_dim_card.withColumn('cc_num',F.col('cc_num').cast(IntegerType()))
slv_dim_card = slv_dim_card.withColumn('expiration_date',F.col('expiration_date').cast(DateType()))
slv_dim_card = slv_dim_card.withColumn('issue_date',F.col('issue_date').cast(DateType()))
slv_dim_card = slv_dim_card.withColumn('credit_limit',F.col('credit_limit').cast(IntegerType()))

In [56]:
slv_dim_card.printSchema()

root
 |-- cc_num: string (nullable = true)
 |-- card_provider: string (nullable = true)
 |-- card_category: string (nullable = true)
 |-- expiration_date: date (nullable = true)
 |-- issue_date: date (nullable = true)
 |-- credit_limit: integer (nullable = true)



In [57]:
slv_dim_card.count()

999

In [58]:
slv_dim_card.dropDuplicates().count()

999

In [60]:
#Checking Nulls
slv_dim_card.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in slv_dim_card.columns]).show()

+------+-------------+-------------+---------------+----------+------------+
|cc_num|card_provider|card_category|expiration_date|issue_date|credit_limit|
+------+-------------+-------------+---------------+----------+------------+
|     0|            0|            0|              0|         0|           0|
+------+-------------+-------------+---------------+----------+------------+



In [ ]:
slv_dim_card.select('card_provider').distinct().show(truncate=False)

+---------------------------+
|card_provider              |
+---------------------------+
|VISA 16 digit              |
|VISA 13 digit              |
|Discover                   |
|Diners Club / Carte Blanche|
|American Express           |
|Maestro                    |
|Mastercard                 |
|JCB 16 digit               |
|VISA 19 digit              |
|JCB 15 digit               |
+---------------------------+



In [62]:
slv_dim_card.select('card_category').distinct().show(truncate=False)

+-------------+
|card_category|
+-------------+
|Platinum     |
|Basic        |
|Gold         |
|Signature    |
|Infinite     |
+-------------+



In [63]:
slv_dim_card.write \
    .format('parquet') \
    .mode('overwrite') \
    .save('s3a://cc-transaction-pipeline/slv/slv_dim_card/')